In [ ]:
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP

In [ ]:
# read in dataframe, only use the columns: reviewText, summary, overall, asin
df = pd.read_csv("apple_preprocessed.csv")
corpus = list(df["corpus"])

In [ ]:
# Using pre-trained BERT model to generate documnet-word embeddings
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = sentence_model.encode(corpus, show_progress_bar=True)

In [ ]:
# Train BERTopic
# default min_topic_size = 10
# default nr_topics = None

umap_model = UMAP(n_neighbors=15, 
                  n_components=10, 
                  metric='cosine', 
                  low_memory=False)

#min_samples
hdbscan_model = HDBSCAN(min_cluster_size=20, # default: 10
                        metric='euclidean', 
                        prediction_data=False) # if set to true, can allow for new predictions later 

topic_model = BERTopic(umap_model=umap_model, 
                       hdbscan_model=hdbscan_model,
                       min_topic_size = 100, # default: 10
                       nr_topics = 20).fit(corpus, embeddings) # defaul: None - auto

In [ ]:
topic_model.visualize_topics()

In [ ]:
# Run the visualization with the original embeddings
topic_model.visualize_documents(corpus, embeddings=embeddings)

In [ ]:
# Reduce dimensionality of embeddings, this step is optional but much faster to perform iteratively:
from umap import UMAP
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
topic_model.visualize_documents(corpus, reduced_embeddings=reduced_embeddings)